# nano Beatrix — the binding's capacity: the key rotation read past the row

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_capacity.ipynb)

**The question**: Turning the told head's eight key pairs by fixed phases of the key's characters (`colab/nano_keyrot.ipynb`, `colab/nano_largerows.ipynb`) binds the whole key: on every model trained so far the head's look finds the asked record with 0 wrong, at 24 and at 48 records a row. A 4,096-byte row holds at most ~92 competing keys. How many keys can eight bound pairs keep apart?

**This notebook** trains nothing. Each stored model reads 192 rows at its own layout and keys' form (half with the key-hard keys); the trunk runs through the told head's block and the head's rotated query and key are kept at every keyed end-marker. Each index entry's true score is then set against pools of other keys: REAL (the other rows' end-markers, to the pool's size) and EMULATED (a pooled key's content unrotated by its own phases and turned by the phases of a fresh key, to 10^6). The leak - one minus the true record's share of the head's softmax over the true end-marker and N competitors -

    leak_N = W_N / (1 + W_N),   W_N = sum_n exp(s_n - s_true)

is recorded at N = 10^2 .. 10^6 as its mean, median, 99th percentile and max over the entries, with the look error (leak >= 1/2) and the argmax error (a competitor at or above the true score), beside the moment of random phases at the model's own margin

    E[leak_N] ~ N I0(s)^8 / (exp(8 s) + N I0(s)^8),   s = the mean key-pair margin / 8.

The moment is an annealed average, dominated by rare near-aligned competitors; the median shows the typical leak.

| models | what | the row |
|---|---|---|
| LRK s1-s6, LRP s1-s6 | the key rotation at 48 records a row, with and without the key-hard rows | 64 slots of 64 bytes, keys of 6-9 bytes |
| LLK s1-s6 | the same with the long keys (a cue word + six digits, the rotation's reach 12) | keys of 9-12 bytes |
| KRK s1-s6, KRP s1-s6 | the key rotation at 24 records a row, with and without the key-hard rows | 32 slots of 128 bytes |

**The cells**: C1 pins the trainer and holds the decision block; C2 writes the held-out documents (the rows' filler), checks every model's weights on the Hub and probes the push path; C3 reads the thirty models one at a time, resume-first, each record shipped to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) `bed/reports/capacity/` as it lands; C4 renders the tables.

**Before C1**: Runtime > Change runtime type > CPU. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on (tokenless, the read runs and ships nothing). Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` ends the session between models; a restarted session resumes where it stopped. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/capacity.py`, `key_rotary.py`, `rotary_scale.py`, `tests/test_capacity.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK (a CPU runtime). RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.85" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 85), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
torch.set_num_threads(os.cpu_count() or 2)
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| cpu threads', torch.get_num_threads(),
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (the read runs on the CPU)')

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out documents, the records, a model's weights while it is read
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # the models' weights are read from here; every record ships here; tokenless = nothing ships
DEST = 'bed/reports/capacity'                          # the records' folder on the Hub
R = 192                                                # rows a model, half key-hard (the plan: 192 unless a model prices over 12 min; the CPU dry run: 0.56 s a row)
SEED = 1                                               # the rows' generator: the same rows for every model of a layout and keys' form
NS = (100, 1000, 10000, 100000, 1000000)               # the competitor counts (REAL: those the pool holds; EMULATED: all, the bank 10^6)
DRAWS = 8                                              # REAL draws an entry at each N
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
FAM = {'LRK': '_cp4_gc48_kh50_va1_pt10b4h0_krt_ahdigit1_largerows',     # 48 records, the key-hard rows
       'LRP': '_cp4_gc48_va1_pt10b4h0_krt_ahdigit1_largerows',          # 48 records
       'LLK': '_cp4_gc48_kl_kh50_va1_pt10b4h0_krt_ahdigit1_largerows',  # 48 records, the long keys
       'KRK': '_cp4_kh50_va1_pt10b4h0_krt_ahdigit1_keyrot',             # 24 records, the key-hard rows
       'KRP': '_cp4_va1_pt10b4h0_krt_ahdigit1_keyrot'}                 # 24 records
CORES = [(f'{f} s{s}', f'alhena-both_s{s}{TAIL}{FAM[f]}') for f in ('LRK', 'LRP', 'LLK', 'KRK', 'KRP') for s in range(1, 7)]
print('decision block:', len(CORES), 'models |', R, 'rows a model, seed', SEED, '| N', NS, '|', DRAWS, 'draws | ships to',
      REPO + '/' + DEST if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the held-out documents (the rows' filler), every model's weights on the Hub, the push probe. Nothing is read yet.
import json, os, time
from huggingface_hub import HfApi
from geolip.nano_beatrix import bed_corpus
HELD = bed_corpus.write_heldout(f'{OUT}/bed')         # the first 2,048 documents of fineweb-edu sample-10BT (the beds' held-out head)
print('held-out documents', HELD, os.path.getsize(HELD) / 1e6, 'MB')
api = HfApi(token=HF_TOKEN or False)
FILES = set(api.list_repo_files(REPO))
missing = [lab for lab, tag in CORES if f'bed/weights/{tag}/key_rot.safetensors' not in FILES]
assert not missing, f'models without their stored phases on the Hub: {missing}'
done_hub = [lab for lab, tag in CORES if f'{DEST}/{tag}_capacity_r{R}_s{SEED}.json' in FILES]
print('every model on the Hub with its phases:', len(CORES), '| records already shipped (done before):', done_hub or 'none')
PUSH_OK = False
if HF_TOKEN:
    path = 'bed/preflight/push_probe_capacity.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'cpu_threads': torch.get_num_threads(),
                       'models': [lab for lab, _ in CORES], 'rows': R}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano capacity: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below read and ship nothing')

In [ ]:
# C3 (R) — THE READ: the thirty models one at a time, resume-first (a record in OUT/capacity or on the Hub is done before); a STOP file in OUT
# starts no new model; each record ships to DEST as it lands (a background thread with retries; the local record is the done mark, and a
# restarted session ships what is local and not on the Hub). The job's size, the time spent and the time left print after every model.
import json, os, shutil, threading, time
from huggingface_hub import hf_hub_download, snapshot_download
from geolip.nano_beatrix import capacity as cp, rotary_scale as rs
held = rs.load_heldout(HELD)
REC = f'{OUT}/capacity'
os.makedirs(REC, exist_ok=True)
FILES = set(api.list_repo_files(REPO))
threads, failed = [], []

def _ship(path):
    for i in range(6):
        try:
            api.upload_file(path_or_fileobj=path, path_in_repo=f'{DEST}/{os.path.basename(path)}', repo_id=REPO,
                            commit_message=f'nano capacity: {os.path.basename(path)[:90]}')
            return
        except Exception as e:
            print('ship retry', i + 1, os.path.basename(path)[:60], type(e).__name__)
            time.sleep(min(300, 10 * 2 ** i))
    failed.append(path)

def ship(path):
    if HF_TOKEN and PUSH_OK:
        t = threading.Thread(target=_ship, args=(path,))
        t.start()
        threads.append(t)

for f in sorted(os.listdir(REC)):                     # local records not on the Hub: shipped first
    if f.endswith('.json') and f'{DEST}/{f}' not in FILES:
        ship(f'{REC}/{f}')
t0, read_n, recs = time.time(), 0, {}
for n, (label, tag) in enumerate(CORES, 1):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new model starts')
        break
    name = cp.record_name(tag, R, SEED)
    local = f'{REC}/{name}'
    if not os.path.exists(local) and f'{DEST}/{name}' in FILES:
        shutil.copy(hf_hub_download(REPO, f'{DEST}/{name}', local_dir=f'{OUT}/hub', token=HF_TOKEN or False), local)
    t1 = time.time()
    if not os.path.exists(local):
        snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN or False)
        cp.run(f'{OUT}/w/bed/weights/{tag}', held, REC, R=R, seed=SEED, ns=NS, draws=DRAWS, log=print)
        ship(local)
        shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)
        read_n += 1
    rec = recs[label] = json.load(open(local, encoding='utf-8'))
    e = rec['emulated']
    spent = time.time() - t0
    left = spent / read_n * (len(CORES) - n) if read_n else 0.0
    print(f"{label} ({n}/{len(CORES)}): s {rec['s_core']:.2f} | mean leak 10^4 {e['10000']['mean']:.1e} (moment {e['10000']['moment']:.1e}), "
          f"10^6 {e['1000000']['mean']:.2f} (moment {e['1000000']['moment']:.2f}) | look error 10^4 {e['10000']['look_error']:.4f}, "
          f"10^6 {e['1000000']['look_error']:.4f} | {time.time() - t1:.0f} s | spent {spent / 60:.1f} min, ~{left / 60:.1f} min left")
for t in threads:
    t.join()
FILES = set(api.list_repo_files(REPO))
print('read this session:', read_n, '| UNSHIPPED:', [f for f in sorted(os.listdir(REC)) if f.endswith('.json') and f'{DEST}/{f}' not in FILES]
      if HF_TOKEN else 'tokenless (nothing ships)')

In [ ]:
# C4 — THE TABLES (the records in OUT/capacity): per model the margin s, the mean (median) leak EMULATED at each N against the moment,
# the look error, and the REAL means where the pool holds N.
from IPython.display import Markdown, display
rows = ['| model | s | N = 10^3: mean (median) / moment | 10^4 | 10^5 | 10^6 | look error 10^4 / 10^6 | REAL mean 10^3 / 10^4 |',
        '|---|---|---|---|---|---|---|---|']
for label, tag in CORES:
    if label not in recs:
        continue
    r = recs[label]
    e = r['emulated']
    f = lambda n: f"{e[n]['mean']:.1e} ({e[n]['median']:.1e}) / {e[n]['moment']:.1e}"
    rr = ' / '.join(f"{r['real'][n]['mean']:.1e}" if n in r['real'] else '-' for n in ('1000', '10000'))
    rows.append(f"| {label} | {r['s_core']:.2f} | {f('1000')} | {f('10000')} | {f('100000')} | {f('1000000')} | "
                f"{e['10000']['look_error']:.4f} / {e['1000000']['look_error']:.4f} | {rr} |")
display(Markdown('\n'.join(rows)))